# JS13 Praktikum 2 - Klasifikasi Iris dengan Neural Network

Praktikum ini membandingkan konfigurasi jumlah neuron hidden layer dan fungsi aktivasi pada klasifikasi dataset Iris. `MLPClassifier` dari scikit-learn digunakan sebagai implementasi neural network yang runnable pada Python 3.14 ketika TensorFlow belum tersedia.

In [5]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print(f"Data training: {X_train.shape}, data testing: {X_test.shape}")

Data training: (120, 4), data testing: (30, 4)


## Fungsi eksperimen

Konfigurasi awal mengikuti modul, yaitu hidden layer `(10, 8)`. Konfigurasi kedua mengubah jumlah neuron menjadi `(16, 12)`. Eksperimen ketiga membandingkan sigmoid dan ReLU dengan arsitektur yang sama.

In [6]:
def run_experiment(name, hidden_layers=(10, 8), activation="relu"):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        solver="lbfgs",
        alpha=1e-4,
        max_iter=1000,
        random_state=42,
    )
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)
    predictions = model.predict(X_test)
    return {
        "Konfigurasi": name,
        "Aktivasi": activation,
        "Hidden layer": str(hidden_layers),
        "Loss (log loss)": log_loss(y_test, probabilities),
        "Akurasi": accuracy_score(y_test, predictions),
        "model": model,
    }


baseline = run_experiment("Baseline", (10, 8), "relu")
hidden_changed = run_experiment("Jumlah neuron diubah", (16, 12), "relu")
sigmoid_result = run_experiment("Sigmoid", (10, 8), "logistic")
relu_result = baseline

results = pd.DataFrame([
    {k: v for k, v in result.items() if k != "model"}
    for result in [baseline, hidden_changed, sigmoid_result]
])
results.style.format({"Loss (log loss)": "{:.6f}", "Akurasi": "{:.2%}"})

,Konfigurasi,Aktivasi,Hidden layer,Loss (log loss),Akurasi
0,Baseline,relu,"(10, 8)",1.834835,93.33%
1,Jumlah neuron diubah,relu,"(16, 12)",0.485333,90.00%
2,Sigmoid,logistic,"(10, 8)",0.442693,93.33%


In [7]:
best = results.loc[results["Akurasi"].idxmax()]
print("Ringkasan hasil:")
print(results.to_string(index=False))
print(f"\nKonfigurasi dengan akurasi tertinggi: {best['Konfigurasi']}")


Ringkasan hasil:
         Konfigurasi Aktivasi Hidden layer  Loss (log loss)  Akurasi
            Baseline     relu      (10, 8)         1.834835 0.933333
Jumlah neuron diubah     relu     (16, 12)         0.485333 0.900000
             Sigmoid logistic      (10, 8)         0.442693 0.933333

Konfigurasi dengan akurasi tertinggi: Baseline


## Kesimpulan

Tabel menunjukkan pengaruh perubahan jumlah neuron hidden layer dan aktivasi terhadap `log loss` serta akurasi. Pada dataset Iris, konfigurasi ReLU dan sigmoid dapat sama-sama menghasilkan akurasi tinggi, tetapi nilai loss dapat berbeda. Hasil dapat sedikit berbeda jika random seed atau pembagian data diubah.